1) Load theknowledge base (raw text, pdf, txt, url) 

In [3]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader
pdfLoader = PyPDFLoader("C:/Users/VK131236/Downloads/python.pdf")
pages = pdfLoader.load()
print(f"Page size: {len(pages)}")

USER_AGENT environment variable not set, consider setting it to identify your requests.


Page size: 164


2) Split the knowlegde base into chunks 

In [4]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50,
    separators= ["\n\n", "\n", ".", ",", ""]
)
 
split_docs = splitter.split_documents(pages)
print(f"Document size: {len(split_docs)}")
 
 

Document size: 1096


3) Create embeddings using chunks and store embeddings into vector database 

In [5]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
embedding_model = OllamaEmbeddings(model="nomic-embed-text:latest")

vector_store=Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory="C:/chroma_db"


)
print(f"Vector store created with type: {type(vector_store)}")    

Vector store created with type: <class 'langchain_community.vectorstores.chroma.Chroma'>


4) Send the prompt along with the knowlegde base as vectors 

In [6]:
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

prompt = PromptTemplate(
    template="Use the following context to answer user questions:\n{context}\nQuestion: {question}",
    input_variables=["context", "question"]
)

llm = ChatOllama(model="llama3.2:latest")

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever(),   
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

5) Question answer 

In [7]:
response= qa_chain({"query": "what is python? "})
print(type(response))
print(response["result"])



C:\Users\VK131236\AppData\Local\Temp\ipykernel_12052\2922197726.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response= qa_chain({"query": "what is python? "})


<class 'dict'>
According to Chapter 1, Section 1.1, Python is a high-level scripting language that can be used for a wide variety of text processing, system administration, and internet-related tasks.


6 . User interface

In [8]:
# STEP 6 - User interface
 
import gradio as gr
 
def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]
 
    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired
 
    return formatted_response
 
# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Gen-AI and Langchain based on the provided text."
)
 
# Launch the interface
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Missing file: C:\Users\VK131236\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\VK131236\.cache\huggingface\gradio\frpc


c:\Users\VK131236\AppData\Local\Programs\Python\Python310\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
